# Train All Models: 4 Attention Mechanisms x 2 Datasets (50% Dataset)

Unified training notebook for retraining all models with RoPE from scratch.

**Models:**
- **MHA-RoPE**: Multi-Head Attention with Rotary Position Embeddings
- **MQA-RoPE**: Multi-Query Attention with RoPE
- **GQA-4-RoPE**: Grouped-Query Attention (4 KV heads) with RoPE
- **MLA**: Multi-Head Latent Attention with Decoupled RoPE

**Datasets:** TinyStories, SimpleStories

**Config:** 256d, 4 layers, 8 heads, vocab 50257, ~16M params each

**Training:** 6000 steps, batch 64 (effective 256), lr 5e-5, BFloat16, cosine schedule

**H100 Optimized:** TF32 enabled, ~35 min per model, ~4.5 hours total

**Dataset:** 50% of each dataset (computed at runtime)

**Output:** 8 checkpoints saved to Google Drive

## 1. Setup & H100 Optimization

In [1]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.2f} GB")
    print(f"BFloat16 supported: {torch.cuda.is_bf16_supported()}")

# H100-specific optimizations
torch.set_float32_matmul_precision('high')  # Enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("\nTF32 optimizations enabled for H100")

Tue Feb 17 13:02:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   32C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [2]:
# Install dependencies
!pip install -q transformers datasets tqdm tensorboard

In [3]:
# Clone repository
import os
if os.path.exists('PROJECT'):
    !rm -rf PROJECT
    print("Existing repo removed")
!git clone -b feat/mla https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
print("Repository cloned")
%cd PROJECT

Cloning into 'PROJECT'...
remote: Enumerating objects: 2153, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 2153 (delta 23), reused 0 (delta 0), pack-reused 2097 (from 1)
Receiving objects: 100% (2153/2153), 14.94 MiB | 530.00 KiB/s, done.
Resolving deltas: 100% (1686/1686), done.
Repository cloned
/content/PROJECT


In [4]:
# Clear cache, patch __init__.py, import all 4 transformer modules
import sys, os, importlib, shutil, torch, gc

print("Clearing Python cache...")
modules_to_remove = [m for m in list(sys.modules.keys())
                     if any(x in m for x in ['mla', 'mqa', 'gqa', 'mha', 'train', 'transformer', 'data_loader', 'attention', 'layers'])]
for module in modules_to_remove:
    del sys.modules[module]
print(f"Removed {len(modules_to_remove)} cached modules")

cache_dirs = [
    '/content/PROJECT/AttentionHeads/mha/__pycache__',
    '/content/PROJECT/AttentionHeads/mqa/__pycache__',
    '/content/PROJECT/AttentionHeads/gqa/__pycache__',
    '/content/PROJECT/AttentionHeads/mla/__pycache__',
    '/content/PROJECT/AttentionHeads/__pycache__',
]
for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

project_root = '/content/PROJECT'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch __init__.py to comment out positional_encoding import
init_file_path = os.path.join(project_root, 'AttentionHeads', 'mha', '__init__.py')
if os.path.exists(init_file_path):
    with open(init_file_path, 'r') as f:
        lines = f.readlines()
    patched_lines = []
    modified = False
    in_block = False
    block_indent = -1
    for line in lines:
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())
        if "from .positional_encoding import" in line:
            in_block = True
            block_indent = indent
            if not stripped.startswith('#'):
                patched_lines.append(f"# PATCHED: {stripped}\n")
                modified = True
            else:
                patched_lines.append(line)
            continue
        if in_block:
            if indent > block_indent or stripped == '' or (')' in stripped and indent <= block_indent):
                if not stripped.startswith('#'):
                    patched_lines.append(f"# PATCHED: {stripped}\n")
                    modified = True
                else:
                    patched_lines.append(line)
                if ')' in stripped and indent <= block_indent:
                    in_block = False
                continue
            else:
                in_block = False
        patched_lines.append(line)
    if modified:
        with open(init_file_path, 'w') as f:
            f.writelines(patched_lines)
        print("Patched __init__.py")

# Import all 4 transformer modules
from AttentionHeads.mha import transformer as mha_transformer
from AttentionHeads.mqa import transformer as mqa_transformer
from AttentionHeads.gqa import transformer as gqa_transformer
from AttentionHeads.mla import transformer as mla_transformer
from AttentionHeads.mha import data_loader, train

for mod in [mha_transformer, mqa_transformer, gqa_transformer, mla_transformer, data_loader, train]:
    importlib.reload(mod)

GPTNeoTrainer = train.GPTNeoTrainer
print("All modules imported successfully!")

Clearing Python cache...
Removed 9 cached modules
All modules imported successfully!


## 2. Configuration

In [5]:
# Pre-compute 50% dataset sizes
from datasets import load_dataset_builder

DATASET_SIZES = {}

print("Computing 50% dataset sizes...")
ts_builder = load_dataset_builder('roneneldan/TinyStories')
ts_train_total = ts_builder.info.splits['train'].num_examples
ts_val_total = ts_builder.info.splits['validation'].num_examples
DATASET_SIZES['tinystories'] = {'train': ts_train_total // 2, 'val': ts_val_total // 2}
print(f"  TinyStories: {DATASET_SIZES['tinystories']['train']:,} train, {DATASET_SIZES['tinystories']['val']:,} val (of {ts_train_total:,} / {ts_val_total:,} total)")

ss_builder = load_dataset_builder('SimpleStories/SimpleStories')
ss_train_total = ss_builder.info.splits['train'].num_examples
ss_val_total = ss_builder.info.splits['test'].num_examples
DATASET_SIZES['simplestories'] = {'train': ss_train_total // 2, 'val': ss_val_total // 2}
print(f"  SimpleStories: {DATASET_SIZES['simplestories']['train']:,} train, {DATASET_SIZES['simplestories']['val']:,} val (of {ss_train_total:,} / {ss_val_total:,} total)")

print("50% dataset sizes computed!")


def get_training_config(model_name, architecture, dataset, log_dir, checkpoint_dir):
    """Create configuration dictionary for training.

    Args:
        model_name: Name for logging (e.g. 'gptneo_mha_ts')
        architecture: Architecture type (e.g. 'gpt_neo_mha')
        dataset: 'tinystories' or 'simplestories'
        log_dir: TensorBoard log directory
        checkpoint_dir: Model checkpoint directory
    """
    # Dataset-specific settings
    if dataset == 'tinystories':
        dataset_name = 'roneneldan/TinyStories'
        text_column = 'text'
        val_split = 'validation'
        ds_short = 'ts'
    else:
        dataset_name = 'SimpleStories/SimpleStories'
        text_column = 'story'
        val_split = 'test'
        ds_short = 'ss'

    # Use pre-computed 50% dataset sizes
    train_samples = DATASET_SIZES[dataset]['train']
    val_samples = DATASET_SIZES[dataset]['val']

    return {
        "model_name": model_name,
        "architecture": architecture,
        "model": {
            "vocab_size": 50257,
            "hidden_size": 256,
            "num_layers": 4,
            "num_heads": 8,
            "intermediate_size": 1024,
            "max_position_embeddings": 256,
            "dropout": 0.2,
            "position_embedding_type": "rope",
            "activation": "gelu",
            "layer_norm_epsilon": 1e-5,
            "initializer_range": 0.02,
            "tie_word_embeddings": True
        },
        "training": {
            "dataset": dataset,
            "train_samples": train_samples,
            "val_samples": val_samples,
            "batch_size": 64,
            "gradient_accumulation_steps": 4,
            "effective_batch_size": 256,
            "max_steps": 6000,
            "warmup_steps": 600,
            "learning_rate": 5e-5,
            "min_learning_rate": 1e-6,
            "lr_scheduler": "cosine",
            "weight_decay": 0.01,
            "gradient_clip": 0.5,
            "use_bf16": True,
            "optimizer": "adamw",
            "adam_beta1": 0.9,
            "adam_beta2": 0.999,
            "adam_epsilon": 1e-8,
            "max_seq_length": 256
        },
        "data": {
            "tokenizer": "gpt2",
            "dataset_name": dataset_name,
            "dataset_config": "default",
            "text_column": text_column,
            "val_split": val_split,
            "streaming": False,
            "num_workers": 4,
            "prefetch_factor": 2,
            "pin_memory": True
        },
        "logging": {
            "log_dir": log_dir,
            "checkpoint_dir": checkpoint_dir,
            "save_every_steps": 2000,
            "eval_every_steps": 1000,
            "log_every_steps": 50,
            "use_tensorboard": True,
            "use_wandb": False,
            "project_name": f"gptneo-{architecture}-{dataset}",
            "run_name": f"{model_name}_256_4l_{ds_short}"
        },
        "evaluation": {
            "eval_batch_size": 32,
            "eval_max_steps": 100,
            "generation_prompts": [
                "Once upon a time",
                "One day, a little girl",
                "In a big forest",
                "There was a happy dog"
            ],
            "generation_max_length": 100,
            "generation_temperature": 0.8,
            "generation_top_k": 50,
            "generation_top_p": 0.95
        },
        "checkpointing": {
            "save_total_limit": 3,
            "save_best_only": False
        },
        "random_seed": 42
    }

print("Configuration function defined!")

Computing 50% dataset sizes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

  TinyStories: 1,059,859 train, 10,995 val (of 2,119,719 / 21,990 total)


README.md: 0.00B [00:00, ?B/s]

  SimpleStories: 1,057,848 train, 10,685 val (of 2,115,696 / 21,371 total)
50% dataset sizes computed!
Configuration function defined!


## 3. Training Helper

In [6]:
import time

training_results = []

def train_model(name, create_fn, config):
    """Train a single model and return the trainer.

    Args:
        name: Display name (e.g. 'MHA-RoPE')
        create_fn: Model creation function (e.g. mha_transformer.create_gptneo_model)
        config: Training configuration dict

    Returns:
        trainer: GPTNeoTrainer instance (for metrics access)
    """
    print("=" * 80)
    print(f"TRAINING: {name}")
    print(f"Dataset: {config['data']['dataset_name']}")
    print("=" * 80)

    # Create model
    model = create_fn(config['model'])
    param_count = model.get_num_params()
    print(f"Parameters: {param_count:,}")

    # Train
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    trainer = GPTNeoTrainer(config, device=device, model=model)

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time

    print(f"\n{name} training complete in {elapsed/60:.1f} minutes")

    # Record results
    training_results.append({
        'name': name,
        'dataset': config['training']['dataset'],
        'params': param_count,
        'time_min': elapsed / 60,
        'checkpoint_dir': config['logging']['checkpoint_dir'],
    })

    # Cleanup
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared")

print("Training helper defined!")

Training helper defined!


## 4. Train MHA on TinyStories (~35 min)

In [7]:
mha_ts_config = get_training_config(
    'gptneo_mha_ts', 'gpt_neo_mha', 'tinystories',
    '../logs/gptneo_mha_tinystories', '../checkpoints/gptneo_mha_tinystories')

train_model('MHA-RoPE (TinyStories)', mha_transformer.create_gptneo_model, mha_ts_config)

# Copy best checkpoint
os.makedirs('/content/models/tinystories', exist_ok=True)
src = '/content/checkpoints/gptneo_mha_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mha_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MHA-RoPE (TinyStories)
Dataset: roneneldan/TinyStories
Parameters: 16,025,344
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 16,025,344
  Non-embedding: 3,159,552
  Embedding: 12,865,792


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Val Loss: 3.1701, Val PPL: 23.81

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little boy named Timmy. Timmy loved to go back to her head and play in the park. One day, she went to the park with a long time, and he could go on the world and the fore...

Prompt: One day, a little girl
Generated: One day, a little girl named Lily would find her friend to play in the door. She smiled and ran to the tree. It was so happy that he had to do.

"Look, this is a friends. I don't see me!" her mom said...

Prompt: In a big forest
Generated: In a big forest, it was a little girl named Lily. She would see it, but it was so happy. She was very scared and decided to play in the door.

Suddenly, Lily went to the water. She knew that they woul...

Prompt: There was a happy dog
Generated: There was a happy dog. The and he had many good of it. He was happy to play and have a special water, but it was very good. He ran to the park and said, "I can be very happy!

Training: 100%|██████████| 6000/6000 [04:31<00:00, 22.13it/s, loss=3.3756, ppl=29.24, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_mha_tinystories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 52.61it/s, loss=3.1701]


Final Val Loss: 3.1701, Final Val PPL: 23.81
Checkpoint saved: ../checkpoints/gptneo_mha_tinystories/final_model.pt

✓ Training completed!

MHA-RoPE (TinyStories) training complete in 4.6 minutes
GPU memory cleared
Checkpoint saved: /content/models/tinystories/best_model_mha_ts_50pct.pt


## 5. Train MQA on TinyStories (~35 min)

In [8]:
mqa_ts_config = get_training_config(
    'gptneo_mqa_ts', 'gpt_neo_mqa', 'tinystories',
    '../logs/gptneo_mqa_tinystories', '../checkpoints/gptneo_mqa_tinystories')

train_model('MQA-RoPE (TinyStories)', mqa_transformer.create_gptneo_model, mqa_ts_config)

src = '/content/checkpoints/gptneo_mqa_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mqa_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MQA-RoPE (TinyStories)
Dataset: roneneldan/TinyStories
Parameters: 15,564,800
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 15,564,800
  Non-embedding: 2,699,008
  Embedding: 12,865,792

Setting up TinyStories datasets...
Loading TinyStories dataset (split=train)...
Sampling 1059859 from 2119719 examples...
Dataset size: 1059859
Loading TinyStories dataset (split=validation)...
Sampling 10995 from 21990 examples...
Dataset size: 10995

Dataset Summary:
  Train samples: 1,059,859
  Val samples: 10,995
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:40<03:20, 24.95it/s, loss=4.0164, ppl=55.50, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:01<00:00, 50.84it/s, loss=3.7937]


Val Loss: 3.7937, Val PPL: 44.42

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little and a little. She't a little big, " she was so happy. It was very boy. He was so so and it and it were her and to her.

M. She in a the big, that the the little sh...

Prompt: One day, a little girl
Generated: One day, a little girl to.

" I was a and, "They very so the little Lily a and not his. Shemy day, the, and, but it was mom. The sun was and be was. One day, I were so very. She was a was a little lit...

Prompt: In a big forest
Generated: In a big forest and had a time and a hemy.

The big mom had a very and the her. He was and it.

"No, " I said, thatmy. She was friends, the girl was a Lily. They so he loved to help.

Whenmy, the frie...

Prompt: There was a happy dog
Generated: There was a happy dog he had a little they. He loved to the big. She he was a girl of.

The and wanted to the little they had, but he was her. They was so happy.

The they fo

...

Val Loss: 3.1650, Val PPL: 23.69

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little boy named Timmy. She wanted to play with it. He was so excited to make a special day. One day, the little girl was a very the sky. One day, she was so happy to lik...

Prompt: One day, a little girl
Generated: One day, a little girl named Lily would find a big room. They were going to eat in the ground. One day, he had a walk at the sky. He had a big other toy and. They wanted to do the old park.

"I can we...

Prompt: In a big forest
Generated: In a big forest, it was a little girl named Lily. She would see it, but it was so happy. She was very scared and decided to play in the door.

Suddenly, her mom thought it was so happy. She was much f...

Prompt: There was a happy dog
Generated: There was a happy dog. The and he had many good of big. He was happy to play and have a special time, but it was very good. He ran up and said, "Don't want to be very happy! 

Training: 100%|██████████| 6000/6000 [04:23<00:00, 22.73it/s, loss=3.3682, ppl=29.03, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_mqa_tinystories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 52.50it/s, loss=3.1650]


Final Val Loss: 3.1650, Final Val PPL: 23.69
Checkpoint saved: ../checkpoints/gptneo_mqa_tinystories/final_model.pt

✓ Training completed!

MQA-RoPE (TinyStories) training complete in 4.4 minutes
GPU memory cleared
Checkpoint saved: /content/models/tinystories/best_model_mqa_ts_50pct.pt


## 6. Train GQA-4 on TinyStories (~35 min)

In [9]:
gqa_ts_config = get_training_config(
    'gptneo_gqa_ts', 'gpt_neo_gqa', 'tinystories',
    '../logs/gptneo_gqa_tinystories', '../checkpoints/gptneo_gqa_tinystories')
gqa_ts_config['model']['num_kv_heads'] = 4

train_model('GQA-4-RoPE (TinyStories)', gqa_transformer.create_gptneo_model, gqa_ts_config)

src = '/content/checkpoints/gptneo_gqa_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_gqa_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: GQA-4-RoPE (TinyStories)
Dataset: roneneldan/TinyStories
Parameters: 15,762,176
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 15,762,176
  Non-embedding: 2,896,384
  Embedding: 12,865,792

Setting up TinyStories datasets...
Loading TinyStories dataset (split=train)...
Sampling 1059859 from 2119719 examples...
Dataset size: 1059859
Loading TinyStories dataset (split=validation)...
Sampling 10995 from 21990 examples...
Dataset size: 10995

Dataset Summary:
  Train samples: 1,059,859
  Val samples: 10,995
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:40<03:24, 24.46it/s, loss=4.0301, ppl=56.26, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 49.44it/s, loss=3.7967]


Val Loss: 3.7967, Val PPL: 44.55

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little girl. He was a big in a big, " she it, they with the very very you. He was a little girl. He it were her and to her.

M, to the big!


Her the was happy and happy ...

Prompt: One day, a little girl
Generated: One day, a little girl to.

" day, " and, "They very, the the Lily a and not his. She and day, the, and, "L and he. The in the girl was was. One day, I have so very. They was a was a little and they, ...

Prompt: In a big forest
Generated: In a big forest and had a her and a hemy. They was to the mom. The very and the his the park and was very.

"", ", said, "my. She said wasmy the girl was the Lily. They to the mom. The said, but thatm...

Prompt: There was a happy dog
Generated: There was a happy dog said, "But they, but a in the big. She he was a girl of. 

One day, mom, "The, said, "Mom, it was to she her and they had. They wanted to the for. The i

...

Val Loss: 3.1824, Val PPL: 24.11

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little boy named Timmy. Timmy loved to go back to her head and play with her toys. One day, she went to the park with a and the bird. He said, "It's a big and his mommy. ...

Prompt: One day, a little girl
Generated: One day, a little girl named Lily. She loved to play with her head and they found a fun of old man. She put on a walk at the sky and said, "Look, Tom, this is a lot of tree. You'll go to her mom and e...

Prompt: In a big forest
Generated: In a big forest, a little girl named Lily. She liked to play in the park in the floor and run home. She knew that the family was scared and had a big sky. 

Lily was so happy that she didn't want to p...

Prompt: There was a happy dog
Generated: There was a happy dog named Timmy. She was a big noise. Timmy loved to play and have a special time, but he had a good new. One day, the big boy was very so happy. He wanted 

Training: 100%|██████████| 6000/6000 [04:27<00:00, 22.45it/s, loss=3.3874, ppl=29.59, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_gqa_tinystories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 52.35it/s, loss=3.1824]


Final Val Loss: 3.1824, Final Val PPL: 24.11
Checkpoint saved: ../checkpoints/gptneo_gqa_tinystories/final_model.pt

✓ Training completed!

GQA-4-RoPE (TinyStories) training complete in 4.5 minutes
GPU memory cleared
Checkpoint saved: /content/models/tinystories/best_model_gqa_ts_50pct.pt


## 7. Train MLA on TinyStories (~35 min)

In [10]:
mla_ts_config = get_training_config(
    'gptneo_mla_ts', 'gpt_neo_mla', 'tinystories',
    '../logs/gptneo_mla_tinystories', '../checkpoints/gptneo_mla_tinystories')
mla_ts_config['model']['d_c'] = 128
mla_ts_config['model']['d_rope'] = 16

train_model('MLA (TinyStories)', mla_transformer.create_gptneo_model, mla_ts_config)

src = '/content/checkpoints/gptneo_mla_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mla_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MLA (TinyStories)
Dataset: roneneldan/TinyStories
Parameters: 16,037,632
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 16,037,632
  Non-embedding: 3,171,840
  Embedding: 12,865,792

Setting up TinyStories datasets...
Loading TinyStories dataset (split=train)...
Sampling 1059859 from 2119719 examples...
Dataset size: 1059859
Loading TinyStories dataset (split=validation)...
Sampling 10995 from 21990 examples...
Dataset size: 10995

Dataset Summary:
  Train samples: 1,059,859
  Val samples: 10,995
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:41<03:25, 24.37it/s, loss=3.8150, ppl=45.38, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:01<00:00, 51.12it/s, loss=3.5927]


Val Loss: 3.5927, Val PPL: 36.33

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little girl named Lily. She was a big, the little girl and she was very happy. He saw a big boy named Lily and it was it were her and was a, he had his mom.


The little ...

Prompt: One day, a little girl
Generated: One day, a little girl to play, but her to go to and, "They said, the little girl!"

"What could it, her.
The little girl was a little girl, in the girl named his. One day, I have to the. She was a wa...

Prompt: In a big forest
Generated: In a big forest and had a big and a hemy.

The boy was happy. He wanted to his the park and was very big.

"my. It said. They ran to go to was, the park. The little little girl. One day, but her mom t...

Prompt: There was a happy dog
Generated: There was a happy dog. She was it had a lot. She was very. She had a big girl of. 

One day, and was very big, but he was very him.


The boy was so they found a he was so. T

...

Val Loss: 3.0710, Val PPL: 21.56

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, there was a little boy named Timmy. Timmy loved to go back to his head and play in the sky. One day, Timmy went to the park with his mom and said, "Mom, I have many of the park. It i...

Prompt: One day, a little girl
Generated: One day, a little girl named Lily would find a big room with her head. They wanted to play with her friend, but he had a walk. She wanted to do. Her mom smiled and said, "I can do you go to the store!...

Prompt: In a big forest
Generated: In a big forest, there was a little girl named Lily. She would see a little girl named Lily. She wanted to play with her mom, but she didn't know, but she saw a big ground.

But then, the rabbit was a...

Prompt: There was a happy dog
Generated: There was a happy dog named Tim and Timmy was a big noise. Timmy loved to play with her friends. It was a little boy with a shiny ground.

The big boy was very sad. They got 

Training: 100%|██████████| 6000/6000 [04:29<00:00, 22.28it/s, loss=3.2780, ppl=26.52, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_mla_tinystories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 50.92it/s, loss=3.0710]


Final Val Loss: 3.0710, Final Val PPL: 21.56
Checkpoint saved: ../checkpoints/gptneo_mla_tinystories/final_model.pt

✓ Training completed!

MLA (TinyStories) training complete in 4.5 minutes
GPU memory cleared
Checkpoint saved: /content/models/tinystories/best_model_mla_ts_50pct.pt


## 8. Save TinyStories Checkpoints to Google Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

drive_ts_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/TinyStories'
os.makedirs(drive_ts_dir, exist_ok=True)

ts_files = [
    'best_model_mha_ts_50pct.pt',
    'best_model_mqa_ts_50pct.pt',
    'best_model_gqa_ts_50pct.pt',
    'best_model_mla_ts_50pct.pt',
]

for f in ts_files:
    src = f'/content/models/tinystories/{f}'
    dst = os.path.join(drive_ts_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  {f} -> Drive ({size_mb:.1f} MB)")
    else:
        print(f"  WARNING: {f} not found")

print(f"\nTinyStories checkpoints saved to: {drive_ts_dir}")

Mounted at /content/drive
  best_model_mha_ts_50pct.pt -> Drive (183.7 MB)
  best_model_mqa_ts_50pct.pt -> Drive (178.5 MB)
  best_model_gqa_ts_50pct.pt -> Drive (180.7 MB)
  best_model_mla_ts_50pct.pt -> Drive (183.7 MB)

TinyStories checkpoints saved to: /content/drive/MyDrive/AttentionHeads_RoPE_50pct/TinyStories


## 9. Train MHA on SimpleStories (~35 min)

In [12]:
mha_ss_config = get_training_config(
    'gptneo_mha_ss', 'gpt_neo_mha', 'simplestories',
    '../logs/gptneo_mha_simplestories', '../checkpoints/gptneo_mha_simplestories')

train_model('MHA-RoPE (SimpleStories)', mha_transformer.create_gptneo_model, mha_ss_config)

os.makedirs('/content/models/simplestories', exist_ok=True)
src = '/content/checkpoints/gptneo_mha_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mha_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MHA-RoPE (SimpleStories)
Dataset: SimpleStories/SimpleStories
Parameters: 16,025,344
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 16,025,344
  Non-embedding: 3,159,552
  Embedding: 12,865,792

Setting up SimpleStories datasets...
Loading SimpleStories dataset (split=train)...


data/train-00000-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

...

Val Loss: 4.6215, Val PPL: 101.65

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, a boy named he felt a eyes. They to a and looked, but she saw a her to to in the voice. The boy was filled of the day, and and the her day. As he reached the world, he felt a treasur...

Prompt: One day, a little girl
Generated: One day, a little girl named Jose felt the, and he thought in the her garden. He was joy in the tree of joy. They looked at the sky, but he looked his other. The. "I must do the her, they, they felt a...

Prompt: In a big forest
Generated: In a big forest, a small girl named,, and she laughed, but the, the little little and had be. The air was the dark and smiled, but she, it would that and the boy thought. The small, she felt a and saw...

Prompt: There was a happy dog
Generated: There was a happy dog, he and they had her world. He would her, and he felt a. With a deep, he had the the friend was a dark small girl. One day, she had the treasure, to th

Training: 100%|██████████| 6000/6000 [04:28<00:00, 22.34it/s, loss=4.7526, ppl=115.89, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_mha_simplestories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 50.93it/s, loss=4.6215]


Final Val Loss: 4.6215, Final Val PPL: 101.65
Checkpoint saved: ../checkpoints/gptneo_mha_simplestories/final_model.pt

✓ Training completed!

MHA-RoPE (SimpleStories) training complete in 4.5 minutes
GPU memory cleared
Checkpoint saved: /content/models/simplestories/best_model_mha_ss_50pct.pt


## 10. Train MQA on SimpleStories (~35 min)

In [13]:
mqa_ss_config = get_training_config(
    'gptneo_mqa_ss', 'gpt_neo_mqa', 'simplestories',
    '../logs/gptneo_mqa_simplestories', '../checkpoints/gptneo_mqa_simplestories')

train_model('MQA-RoPE (SimpleStories)', mqa_transformer.create_gptneo_model, mqa_ss_config)

src = '/content/checkpoints/gptneo_mqa_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mqa_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MQA-RoPE (SimpleStories)
Dataset: SimpleStories/SimpleStories
Parameters: 15,564,800
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 15,564,800
  Non-embedding: 2,699,008
  Embedding: 12,865,792

Setting up SimpleStories datasets...
Loading SimpleStories dataset (split=train)...
Sampling 1057848 from 2115696 examples...
Dataset size: 1057848
Loading SimpleStories dataset (split=test)...
Sampling 10685 from 21371 examples...
Dataset size: 10685

Dataset Summary:
  Train samples: 1,057,848
  Val samples: 10,685
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:40<03:26, 24.18it/s, loss=4.9531, ppl=141.61, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 49.02it/s, loss=4.8127]


Val Loss: 4.8127, Val PPL: 123.07

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time on of. " could a a,. A. They as a the, she it she and their. The had you. 

" and. He it to her, to a. 

 the old in a the her. "I the was she of and in the their, could his a. They w...

Prompt: One day, a little girl
Generated: One day, a little girl to. The, her to a had and,. She. The the the his a and not his. "I, her.


, it and he. The in the girl, was. One, the, but the girl. With but a was?" her, they, that. It she a ...

Prompt: In a big forest
Generated: In a big forest and had they her. The he a to. "What and to had, but and their she the had his. They. 

As of a and said. "I. She felt was, the he was the with. They, for it. The. The the that, a the....

Prompt: There was a happy dog
Generated: There was a happy dog he.

 they, the a in the had. In he, he, her. "I the felt the the old, they him,, he, she. The,, she her. The. He found a he, could. The it on the with

...

Val Loss: 4.5625, Val PPL: 95.82

Generating samples...


Training: 100%|██████████| 6000/6000 [04:23<00:00, 22.80it/s, loss=4.7018, ppl=110.15, lr=1.00e-06]



Prompt: Once upon a time
Generated: Once upon a time, a boy named he felt a eyes. One day, the girl, it was a small and to to the wind. As he opened the stars, she could help a and the her day. As he reached the world, he felt a treasur...

Prompt: One day, a little girl
Generated: One day, a little girl named he felt the, and he thought in the her garden. He looked to in the tree of joy. They looked at the sky, but he looked his other. The. They could be a tree in the sun, they...

Prompt: In a big forest
Generated: In a big forest, a small sun named she had a story. They were a, and it was a girl. Each day, she had a old friends in a treasure, but she saw a soft a way to it. They knew the heart was a small treas...

Prompt: There was a happy dog
Generated: There was a happy dog, he was a old her world. He found a, and he felt a. With a deep, he had the the friend was a dark small girl. One day, she had a treasure, to the old other. The girl of his. The ...

Checkpoint saved: ../chec

Evaluating: 100%|██████████| 100/100 [00:01<00:00, 51.18it/s, loss=4.5625]


Final Val Loss: 4.5625, Final Val PPL: 95.82
Checkpoint saved: ../checkpoints/gptneo_mqa_simplestories/final_model.pt

✓ Training completed!

MQA-RoPE (SimpleStories) training complete in 4.4 minutes
GPU memory cleared
Checkpoint saved: /content/models/simplestories/best_model_mqa_ss_50pct.pt


## 11. Train GQA-4 on SimpleStories (~35 min)

In [14]:
gqa_ss_config = get_training_config(
    'gptneo_gqa_ss', 'gpt_neo_gqa', 'simplestories',
    '../logs/gptneo_gqa_simplestories', '../checkpoints/gptneo_gqa_simplestories')
gqa_ss_config['model']['num_kv_heads'] = 4

train_model('GQA-4-RoPE (SimpleStories)', gqa_transformer.create_gptneo_model, gqa_ss_config)

src = '/content/checkpoints/gptneo_gqa_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_gqa_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: GQA-4-RoPE (SimpleStories)
Dataset: SimpleStories/SimpleStories
Parameters: 15,762,176
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 15,762,176
  Non-embedding: 2,896,384
  Embedding: 12,865,792

Setting up SimpleStories datasets...
Loading SimpleStories dataset (split=train)...
Sampling 1057848 from 2115696 examples...
Dataset size: 1057848
Loading SimpleStories dataset (split=test)...
Sampling 10685 from 21371 examples...
Dataset size: 10685

Dataset Summary:
  Train samples: 1,057,848
  Val samples: 10,685
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:40<03:22, 24.66it/s, loss=4.9918, ppl=147.20, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 49.20it/s, loss=4.8668]


Val Loss: 4.8668, Val PPL: 129.91

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time on of. " to a a,. A. He was a the, she, she and their. The!" he. 

",. He it. She, he,,. A of the to in a the her. "I the was she, and in the their, she to the. They with her, she, he...

Prompt: One day, a little girl
Generated: One day, a little girl to. The, her to a had and,. She. The the the, a and not his. "I, her. The and, but it and he. The in the girl, was. One the. The, and she. With but a was in her, found, that.


...

Prompt: In a big forest
Generated: In a big forest and had they her. The, a to. The to and to had,, and their she the had his. They. 

As of a, said. "I. She felt was, the he was the,. They, for it. The. The the that,. He. "I she felt ...

Prompt: There was a happy dog
Generated: There was a happy dog he thought.

, the a in the had. In he, he, her. "I the felt the the her, they him,, he, she. The,, she her. The. He found a he, for. The it on the,, s

...

Val Loss: 4.6554, Val PPL: 105.16

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, a boy named he felt a it. One day, the girl, it was a boy and to to in the voice. The boy was a stars of the heart, and the her day. As he would the world and of the her treasure. Su...

Prompt: One day, a little girl
Generated: One day, a little girl felt a map the, and he thought in the her garden. He was a old, with this. "I are you in that?" he said. The girl had the. "I must do the her, they, they felt a hidden, and she ...

Prompt: In a big forest
Generated: In a big forest, a small sun found a girl. It was a big boy, and it was a girl. Each day, she had a old friends in a treasure, but a girl began to a bright day. One day, the a and saw a treasure and w...

Prompt: There was a happy dog
Generated: There was a happy dog, he was a old her world. He was a,, but he felt. With a deep, he had the the in the night.

The big boy found a. They was a small in his other. The gir

Training: 100%|██████████| 6000/6000 [04:25<00:00, 22.57it/s, loss=4.7931, ppl=120.68, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_gqa_simplestories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 50.49it/s, loss=4.6554]


Final Val Loss: 4.6554, Final Val PPL: 105.16
Checkpoint saved: ../checkpoints/gptneo_gqa_simplestories/final_model.pt

✓ Training completed!

GQA-4-RoPE (SimpleStories) training complete in 4.5 minutes
GPU memory cleared
Checkpoint saved: /content/models/simplestories/best_model_gqa_ss_50pct.pt


## 12. Train MLA on SimpleStories (~35 min)

In [15]:
mla_ss_config = get_training_config(
    'gptneo_mla_ss', 'gpt_neo_mla', 'simplestories',
    '../logs/gptneo_mla_simplestories', '../checkpoints/gptneo_mla_simplestories')
mla_ss_config['model']['d_c'] = 128
mla_ss_config['model']['d_rope'] = 16

train_model('MLA (SimpleStories)', mla_transformer.create_gptneo_model, mla_ss_config)

src = '/content/checkpoints/gptneo_mla_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mla_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

TRAINING: MLA (SimpleStories)
Dataset: SimpleStories/SimpleStories
Parameters: 16,037,632
Using device: cuda
Mixed precision: BFloat16

Model Parameters:
  Total: 16,037,632
  Non-embedding: 3,171,840
  Embedding: 12,865,792

Setting up SimpleStories datasets...
Loading SimpleStories dataset (split=train)...
Sampling 1057848 from 2115696 examples...
Dataset size: 1057848
Loading SimpleStories dataset (split=test)...
Sampling 10685 from 21371 examples...
Dataset size: 10685

Dataset Summary:
  Train samples: 1,057,848
  Val samples: 10,685
  Max sequence length: 256
  Batch size: 64
  Vocab size: 50257


Starting training for 6000 steps...
Effective batch size: 256
Gradient accumulation steps: 4


Training:  17%|█▋        | 997/6000 [00:41<03:27, 24.16it/s, loss=4.8048, ppl=122.10, lr=4.93e-05]



Evaluation at step 1000...



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 47.55it/s, loss=4.6573]


Val Loss: 4.6573, Val PPL: 105.35

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time on of the a her and a,. A a a heart, the you she it. The with the was!" you. 

"I. He it were her, he heart, he with of the old in a the world. "I the was she of the in the in, she to...

Prompt: One day, a little girl
Generated: One day, a little girl to a, a her to a you and,. She felt a the the his a and not his. "I, her.


As it and he. The in the girl, was. One the. The a had a. With a a was in the and they, that. It she ...

Prompt: In a big forest
Generated: In a big forest and had they her. The he a boy. He to and to had, but and their she the had his. They to a. He wanted of a and said. 


As was, the he was the stars. They, for a felt. As but the that,...

Prompt: There was a happy dog
Generated: There was a happy dog he thought. The they, but a in the night. In he, he, her. "I that with the the air, they had,, he, she. They,, she her. The had had found a he was for.

...

Val Loss: 4.3567, Val PPL: 78.00

Generating samples...

Prompt: Once upon a time
Generated: Once upon a time, a boy named he felt a young boy named Kim. She looked, but she saw a her to help in the friends. The boy was filled with the day, and and the her eyes thought. He would find the sky ...

Prompt: One day, a little girl
Generated: One day, a little girl named Jose felt the sun. He thought in the her garden, in a bright old man with a. "I are you in the day. They looked on her friends and. "I must do the her, they, they could he...

Prompt: In a big forest
Generated: In a big forest, a girl named Mia, a deep light. They were a, and the world was not be. The air was the dark and stepped in a treasure, but she saw a soft moment. The. One day, the tree was a small tr...

Prompt: There was a happy dog
Generated: There was a happy dog, he was a old air. It was joy, and the treasure felt and. With a deep, he had the the city was a dark and felt the big boy. "Why must be you?" the woman

Training: 100%|██████████| 6000/6000 [04:29<00:00, 22.24it/s, loss=4.4963, ppl=89.69, lr=1.00e-06]


Checkpoint saved: ../checkpoints/gptneo_mla_simplestories/checkpoint_step_6000.pt


Final evaluation...


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 49.21it/s, loss=4.3567]


Final Val Loss: 4.3567, Final Val PPL: 78.00
Checkpoint saved: ../checkpoints/gptneo_mla_simplestories/final_model.pt

✓ Training completed!

MLA (SimpleStories) training complete in 4.5 minutes
GPU memory cleared
Checkpoint saved: /content/models/simplestories/best_model_mla_ss_50pct.pt


## 13. Save SimpleStories Checkpoints to Google Drive

In [16]:
drive_ss_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/SimpleStories'
os.makedirs(drive_ss_dir, exist_ok=True)

ss_files = [
    'best_model_mha_ss_50pct.pt',
    'best_model_mqa_ss_50pct.pt',
    'best_model_gqa_ss_50pct.pt',
    'best_model_mla_ss_50pct.pt',
]

for f in ss_files:
    src = f'/content/models/simplestories/{f}'
    dst = os.path.join(drive_ss_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  {f} -> Drive ({size_mb:.1f} MB)")
    else:
        print(f"  WARNING: {f} not found")

print(f"\nSimpleStories checkpoints saved to: {drive_ss_dir}")

  best_model_mha_ss_50pct.pt -> Drive (183.7 MB)
  best_model_mqa_ss_50pct.pt -> Drive (178.5 MB)
  best_model_gqa_ss_50pct.pt -> Drive (180.7 MB)
  best_model_mla_ss_50pct.pt -> Drive (183.7 MB)

SimpleStories checkpoints saved to: /content/drive/MyDrive/AttentionHeads_RoPE_50pct/SimpleStories


## 14. Save TensorBoard Logs to Google Drive

In [17]:
drive_logs_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/logs'
os.makedirs(drive_logs_dir, exist_ok=True)

log_dirs = [
    'gptneo_mha_tinystories',
    'gptneo_mqa_tinystories',
    'gptneo_gqa_tinystories',
    'gptneo_mla_tinystories',
    'gptneo_mha_simplestories',
    'gptneo_mqa_simplestories',
    'gptneo_gqa_simplestories',
    'gptneo_mla_simplestories',
]

for log_name in log_dirs:
    src = f'/content/logs/{log_name}'
    dst = os.path.join(drive_logs_dir, log_name)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {log_name} -> Drive")
    else:
        print(f"  WARNING: {log_name} not found")

print(f"\nTensorBoard logs saved to: {drive_logs_dir}")

  gptneo_mha_tinystories -> Drive
  gptneo_mqa_tinystories -> Drive
  gptneo_gqa_tinystories -> Drive
  gptneo_mla_tinystories -> Drive
  gptneo_mha_simplestories -> Drive
  gptneo_mqa_simplestories -> Drive
  gptneo_gqa_simplestories -> Drive
  gptneo_mla_simplestories -> Drive

TensorBoard logs saved to: /content/drive/MyDrive/AttentionHeads_RoPE_50pct/logs


## 15. Final Summary

In [18]:
total_time = sum(r['time_min'] for r in training_results)

print("TRAINING SUMMARY")
print("=" * 90)
print(f"{'Model':<30} {'Dataset':<15} {'Params':>12} {'Time (min)':>12} {'Checkpoint':>18}")
print("-" * 90)

for r in training_results:
    print(f"{r['name']:<30} {r['dataset']:<15} {r['params']:>12,} {r['time_min']:>12.1f} {'saved':>18}")

print("-" * 90)
print(f"{'TOTAL':>57} {total_time:>12.1f}")
print("=" * 90)

# List all saved files
print("\nSaved Files:")
print("-" * 60)

drive_base = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct'
for root, dirs, files in os.walk(drive_base):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, drive_base)
        print(f"  {rel_path:<50} {size_mb:>8.2f} MB")

print(f"\nAll 8 models trained and saved in {total_time:.1f} minutes ({total_time/60:.1f} hours)")
print(f"Checkpoints: {drive_base}/TinyStories/ and {drive_base}/SimpleStories/")
print(f"Logs: {drive_base}/logs/")

TRAINING SUMMARY
Model                          Dataset               Params   Time (min)         Checkpoint
------------------------------------------------------------------------------------------
MHA-RoPE (TinyStories)         tinystories       16,025,344          4.6              saved
MQA-RoPE (TinyStories)         tinystories       15,564,800          4.4              saved
GQA-4-RoPE (TinyStories)       tinystories       15,762,176          4.5              saved
MLA (TinyStories)              tinystories       16,037,632          4.5              saved
MHA-RoPE (SimpleStories)       simplestories     16,025,344          4.5              saved
MQA-RoPE (SimpleStories)       simplestories     15,564,800          4.4              saved
GQA-4-RoPE (SimpleStories)     simplestories     15,762,176          4.5              saved
MLA (SimpleStories)            simplestories     16,037,632          4.5              saved
----------------------------------------------------------------

In [19]:
import os
from google.colab import files

# Ensure we are in the base content directory to zip correctly
%cd /content

print("Zipping all checkpoints and logs... This may take a minute.")
# Zip models and logs directories into a single archive
# -r: recursive, -q: quiet
!zip -r -q all_training_artifacts.zip models logs

file_path = '/content/all_training_artifacts.zip'
if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"\nZip file created: {file_path}")
    print(f"Total Size: {size_mb:.2f} MB")
    print("Starting download... (Check your browser's download manager)")
    files.download(file_path)
else:
    print("Error: Zip file was not created. Please check if 'models' and 'logs' directories exist.")

/content
Zipping all checkpoints and logs... This may take a minute.

Zip file created: /content/all_training_artifacts.zip
Total Size: 901.43 MB
Starting download... (Check your browser's download manager)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>